In [3]:
!pip -q install pyspark
from google.colab import files
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
from pyspark import StorageLevel
spark = SparkSession.builder.appName("StreamingSet4").getOrCreate()
files.upload()  # Select all six CSV files
def read_csv(name):
    return spark.read.csv(name, header=True, inferSchema=True)
sessions_df = read_csv("watch_sessions_80000.csv")
extra_df = read_csv("watch_sessions_extra_10000.csv")
subscribers_df = read_csv("subscriber_master_2400.csv")
content_df = read_csv("content_catalog_120.csv")
devices_df = read_csv("device_master_8.csv")
plans_df = read_csv("subscription_plans_4.csv")
print("Main sessions:", sessions_df.count())
sessions_df.printSchema()
sessions_df.show(5, truncate=False)

Saving content_catalog_120.csv to content_catalog_120.csv
Saving device_master_8.csv to device_master_8.csv
Saving subscriber_master_2400.csv to subscriber_master_2400.csv
Saving subscription_plans_4.csv to subscription_plans_4.csv
Saving watch_sessions_80000.csv to watch_sessions_80000.csv
Saving watch_sessions_extra_10000.csv to watch_sessions_extra_10000.csv
Main sessions: 80000
root
 |-- session_id: string (nullable = true)
 |-- subscriber_id: string (nullable = true)
 |-- content_id: string (nullable = true)
 |-- device_id: string (nullable = true)
 |-- session_start: timestamp (nullable = true)
 |-- watch_minutes: integer (nullable = true)
 |-- completion_pct: double (nullable = true)
 |-- buffering_seconds: double (nullable = true)
 |-- stream_quality: string (nullable = true)
 |-- session_status: string (nullable = true)
 |-- data_used_mb: double (nullable = true)
 |-- user_rating: double (nullable = true)

+----------+-------------+----------+---------+-------------------+----

In [4]:
# Make sure to run the setup cell above (O4auvch8Zati) first so sessions_df is loaded in memory!
sessions_df.printSchema()
print("Total Rows:", sessions_df.count())
sessions_df.show(10, truncate=False)

root
 |-- session_id: string (nullable = true)
 |-- subscriber_id: string (nullable = true)
 |-- content_id: string (nullable = true)
 |-- device_id: string (nullable = true)
 |-- session_start: timestamp (nullable = true)
 |-- watch_minutes: integer (nullable = true)
 |-- completion_pct: double (nullable = true)
 |-- buffering_seconds: double (nullable = true)
 |-- stream_quality: string (nullable = true)
 |-- session_status: string (nullable = true)
 |-- data_used_mb: double (nullable = true)
 |-- user_rating: double (nullable = true)

Total Rows: 80000
+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|session_id|subscriber_id|content_id|device_id|session_start      |watch_minutes|completion_pct|buffering_seconds|stream_quality|session_status|data_used_mb|user_rating|
+----------+-------------+----------+---------+-------------------+-------------+--------------+--

In [5]:
from pyspark.sql.functions import col

sessions_df.select(
    "session_id",
    "subscriber_id",
    "content_id",
    "watch_minutes",
    "session_status"
).filter(
    (col("session_status") == "Completed") &
    (col("watch_minutes") > 60)
).show(truncate=False)

+----------+-------------+----------+-------------+--------------+
|session_id|subscriber_id|content_id|watch_minutes|session_status|
+----------+-------------+----------+-------------+--------------+
|WS00000049|SUB01859     |CT0075    |125          |Completed     |
|WS00000179|SUB00935     |CT0054    |62           |Completed     |
|WS00000184|SUB00072     |CT0119    |91           |Completed     |
|WS00000222|SUB01603     |CT0103    |116          |Completed     |
|WS00000233|SUB01233     |CT0027    |158          |Completed     |
|WS00000244|SUB00985     |CT0048    |140          |Completed     |
|WS00000249|SUB00681     |CT0100    |120          |Completed     |
|WS00000257|SUB02115     |CT0120    |68           |Completed     |
|WS00000292|SUB01068     |CT0048    |141          |Completed     |
|WS00000326|SUB00368     |CT0096    |97           |Completed     |
|WS00000415|SUB01758     |CT0105    |150          |Completed     |
|WS00000419|SUB00367     |CT0036    |99           |Completed  

In [6]:
from pyspark.sql.functions import col, when

sessions_enriched = sessions_df.withColumn(
    "watch_hours",
    col("watch_minutes") / 60
).withColumn(
    "high_engagement",
    when(col("completion_pct").isNull(), None)
    .when(col("completion_pct") >= 75, "Yes")
    .otherwise("No")
)

sessions_enriched.select(
    "session_id",
    "watch_minutes",
    "watch_hours",
    "completion_pct",
    "high_engagement"
).show(truncate=False)

+----------+-------------+-------------------+--------------+---------------+
|session_id|watch_minutes|watch_hours        |completion_pct|high_engagement|
+----------+-------------+-------------------+--------------+---------------+
|WS00000001|6            |0.1                |11.8          |No             |
|WS00000002|31           |0.5166666666666667 |56.4          |No             |
|WS00000003|29           |0.48333333333333334|63.0          |No             |
|WS00000004|69           |1.15               |75.0          |Yes            |
|WS00000005|36           |0.6                |23.4          |No             |
|WS00000006|20           |0.3333333333333333 |19.8          |No             |
|WS00000007|78           |1.3                |83.9          |Yes            |
|WS00000008|22           |0.36666666666666664|66.7          |No             |
|WS00000009|21           |0.35               |80.8          |Yes            |
|WS00000010|21           |0.35               |65.6          |No 

In [7]:
sessions_df.sort(
    col("data_used_mb").desc()
).show(20, truncate=False)

+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|session_id|subscriber_id|content_id|device_id|session_start      |watch_minutes|completion_pct|buffering_seconds|stream_quality|session_status|data_used_mb|user_rating|
+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|WS00051169|SUB00155     |CT0069    |DV08     |2026-07-20 00:11:00|167          |97.7          |9.0              |4K            |Completed     |8881.8      |3.0        |
|WS00074513|SUB00684     |CT0056    |DV05     |2026-08-09 21:53:00|157          |96.9          |8.0              |4K            |Completed     |8408.4      |NULL       |
|WS00039201|SUB01664     |CT0027    |DV08     |2026-05-16 06:35:00|161          |97.0          |9.0              |4K            |Completed     |8282.0

In [8]:
from pyspark.sql.functions import col, sum, when

sessions_df.select(
    sum(when(col("subscriber_id").isNull(), 1).otherwise(0)).alias("subscriber_id_nulls"),
    sum(when(col("device_id").isNull(), 1).otherwise(0)).alias("device_id_nulls"),
    sum(when(col("completion_pct").isNull(), 1).otherwise(0)).alias("completion_pct_nulls"),
    sum(when(col("buffering_seconds").isNull(), 1).otherwise(0)).alias("buffering_seconds_nulls"),
    sum(when(col("user_rating").isNull(), 1).otherwise(0)).alias("user_rating_nulls")
).show()

+-------------------+---------------+--------------------+-----------------------+-----------------+
|subscriber_id_nulls|device_id_nulls|completion_pct_nulls|buffering_seconds_nulls|user_rating_nulls|
+-------------------+---------------+--------------------+-----------------------+-----------------+
|                951|           2003|                3513|                   4424|            77605|
+-------------------+---------------+--------------------+-----------------------+-----------------+



In [9]:
df = sessions_df.na.fill(
    {"buffering_seconds": 0}
)
df.show(10, truncate=False)

+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|session_id|subscriber_id|content_id|device_id|session_start      |watch_minutes|completion_pct|buffering_seconds|stream_quality|session_status|data_used_mb|user_rating|
+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|WS00000001|SUB00157     |CT0097    |DV02     |2026-04-26 18:59:00|6            |11.8          |29.0             |HD            |Abandoned     |72.4        |NULL       |
|WS00000002|SUB01430     |CT0031    |DV04     |2026-07-30 06:26:00|31           |56.4          |31.0             |HD            |Interrupted   |422.7       |NULL       |
|WS00000003|SUB01892     |CT0044    |DV08     |2026-02-22 20:55:00|29           |63.0          |26.0             |HD            |Interrupted   |370.0 

In [10]:
df = sessions_df.filter(
    col("device_id").isNotNull()
)

print("Original Count:", sessions_df.count())
print("Without Missing device_id:", df.count())
print("Removed Records:",
      sessions_df.count() - df.count())

Original Count: 80000
Without Missing device_id: 77997
Removed Records: 2003


In [11]:
from pyspark.sql.functions import when, col

sessions_labeled_df = sessions_df.withColumn(
    "completion_level",
    when(col("completion_pct").isNull(), "Unknown")
    .when(col("completion_pct") < 30, "Low")
    .when(col("completion_pct") < 90, "Medium")
    .otherwise("High")
)

sessions_labeled_df.select(
    "session_id",
    "completion_pct",
    "completion_level"
).show(truncate=False)

+----------+--------------+----------------+
|session_id|completion_pct|completion_level|
+----------+--------------+----------------+
|WS00000001|11.8          |Low             |
|WS00000002|56.4          |Medium          |
|WS00000003|63.0          |Medium          |
|WS00000004|75.0          |Medium          |
|WS00000005|23.4          |Low             |
|WS00000006|19.8          |Low             |
|WS00000007|83.9          |Medium          |
|WS00000008|66.7          |Medium          |
|WS00000009|80.8          |Medium          |
|WS00000010|65.6          |Medium          |
|WS00000011|36.1          |Medium          |
|WS00000012|78.6          |Medium          |
|WS00000013|51.6          |Medium          |
|WS00000014|28.1          |Low             |
|WS00000015|16.4          |Low             |
|WS00000016|28.0          |Low             |
|WS00000017|84.3          |Medium          |
|WS00000018|83.3          |Medium          |
|WS00000019|90.3          |High            |
|WS0000002

In [12]:
from pyspark.sql.functions import count, sum

sessions_df.join(
    content_df,
    on="content_id",
    how="inner"
).groupBy(
    "genre"
).agg(
    count("session_id").alias("session_count"),
    sum("watch_minutes").alias("total_watch_minutes")
).orderBy(
    "total_watch_minutes",
    ascending=False
).show(truncate=False)

+-----------+-------------+-------------------+
|genre      |session_count|total_watch_minutes|
+-----------+-------------+-------------------+
|Adventure  |15561        |839700             |
|Drama      |10733        |519872             |
|Animation  |11886        |490335             |
|Comedy     |9105         |452563             |
|Sci-Fi     |9344         |433946             |
|Sports     |9967         |383044             |
|Documentary|7432         |256626             |
|Thriller   |5972         |248430             |
+-----------+-------------+-------------------+



In [13]:
from pyspark.sql.functions import avg

sessions_df.join(
    devices_df,
    on="device_id",
    how="inner"
).groupBy(
    "device_type"
).agg(
    avg("buffering_seconds").alias("avg_buffering_seconds"),
    avg("watch_minutes").alias("avg_watch_minutes")
).orderBy(
    "avg_watch_minutes",
    ascending=False
).show(truncate=False)

+---------------+---------------------+------------------+
|device_type    |avg_buffering_seconds|avg_watch_minutes |
+---------------+---------------------+------------------+
|Laptop Browser |36.930938301907844   |45.84465713083719 |
|Desktop Browser|36.02889125799574    |45.571818733562615|
|Tablet         |35.23614616481949    |45.38943894389439 |
|Smart TV       |36.67392739273927    |45.33776623376623 |
|iPhone         |35.761106907361366   |45.226752503576535|
|Android Phone  |35.676486693680864   |45.0885616509385  |
|Streaming Stick|36.320860309448044   |45.08570541351827 |
|Game Console   |35.51243726816496    |45.04469158686026 |
+---------------+---------------------+------------------+



In [14]:
from pyspark.sql.functions import count, sum, when, round

sessions_df.filter(
    col("subscriber_id").isNotNull()
).join(
    subscribers_df,
    "subscriber_id",
    "inner"
).join(
    plans_df,
    "plan_id",
    "inner"
).groupBy(
    "plan_name"
).agg(
    count("*").alias("total_sessions"),
    sum(
        when(col("session_status") == "Completed", 1)
        .otherwise(0)
    ).alias("completed_sessions")
).withColumn(
    "completion_rate_pct",
    round(
        col("completed_sessions") * 100.0 /
        col("total_sessions"),
        2
    )
).orderBy(
    col("completion_rate_pct").desc()
).show(truncate=False)

+---------+--------------+------------------+-------------------+
|plan_name|total_sessions|completed_sessions|completion_rate_pct|
+---------+--------------+------------------+-------------------+
|Mobile   |17510         |1429              |8.16               |
|Basic    |25086         |2006              |8.0                |
|Premium  |12198         |975               |7.99               |
|Standard |23684         |1868              |7.89               |
+---------+--------------+------------------+-------------------+



In [16]:
from pyspark.sql.functions import count, sum, col

subscribers_df.filter(
    col("subscriber_status") == "Active"
).join(
    plans_df,
    "plan_id",
    "inner"
).groupBy(
    "plan_name"
).agg(
    count("subscriber_id").alias("active_subscribers"),
    sum("monthly_fee_inr").alias("estimated_mrr")
).orderBy(
    col("estimated_mrr").desc()
).show(truncate=False)

+---------+------------------+-------------+
|plan_name|active_subscribers|estimated_mrr|
+---------+------------------+-------------+
|Standard |604               |240996       |
|Premium  |326               |211574       |
|Basic    |632               |157368       |
|Mobile   |443               |66007        |
+---------+------------------+-------------+



In [17]:
unmatched_subscribers = sessions_df.filter(
    col("subscriber_id").isNotNull()
).join(
    subscribers_df,
    "subscriber_id",
    "left_anti"
)

print(
    "Unmatched Subscriber IDs:",
    unmatched_subscribers.count()
)

Unmatched Subscriber IDs: 571


In [18]:
unmatched_devices = sessions_df.filter(
    col("device_id").isNotNull()
).join(
    devices_df,
    "device_id",
    "left_anti"
)

print(
    "Unmatched Device IDs:",
    unmatched_devices.count()
)

Unmatched Device IDs: 481


In [19]:
from pyspark.sql.functions import to_timestamp, year, month, count, sum

df = sessions_df.withColumn(
    "session_start_ts",
    to_timestamp("session_start")
)

df.filter(
    year("session_start_ts") == 2026
).groupBy(
    month("session_start_ts").alias("month")
).agg(
    count("session_id").alias("session_count"),
    sum("watch_minutes").alias("total_watch_minutes")
).orderBy(
    "month"
).show()


+-----+-------------+-------------------+
|month|session_count|total_watch_minutes|
+-----+-------------+-------------------+
|    1|         9070|             407215|
|    2|         8218|             372726|
|    3|         9019|             403971|
|    4|         8817|             399432|
|    5|         9006|             407386|
|    6|         8776|             404203|
|    7|         9204|             416104|
|    8|         8996|             407374|
|    9|         8894|             406105|
+-----+-------------+-------------------+



In [20]:
from pyspark.sql.functions import datediff

df2 = df.join(
    subscribers_df.select(
        "subscriber_id",
        "signup_date"
    ),
    "subscriber_id",
    "left"
).withColumn(
    "days_since_signup",
    datediff(
        "session_start_ts",
        "signup_date"
    )
)

df2.orderBy(
    "session_start_ts"
).select(
    "session_id",
    "subscriber_id",
    "signup_date",
    "session_start_ts",
    "days_since_signup"
).show(10, truncate=False)

+----------+-------------+-----------+-------------------+-----------------+
|session_id|subscriber_id|signup_date|session_start_ts   |days_since_signup|
+----------+-------------+-----------+-------------------+-----------------+
|WS00035235|SUB01707     |2024-02-13 |2026-01-01 00:00:00|688              |
|WS00074739|SUB02206     |2025-08-31 |2026-01-01 00:04:00|123              |
|WS00032058|SUB01248     |2024-03-17 |2026-01-01 00:04:00|655              |
|WS00048560|SUB02342     |2025-08-25 |2026-01-01 00:14:00|129              |
|WS00067087|SUB00563     |2024-08-11 |2026-01-01 00:15:00|508              |
|WS00004245|SUB00711     |2024-10-26 |2026-01-01 00:18:00|432              |
|WS00041710|SUB00261     |2025-05-15 |2026-01-01 00:21:00|231              |
|WS00045132|SUB02167     |2025-11-22 |2026-01-01 00:28:00|40               |
|WS00036438|SUB01237     |2024-02-01 |2026-01-01 00:41:00|700              |
|WS00030050|SUB00646     |2024-11-20 |2026-01-01 00:51:00|407              |

In [21]:
from pyspark.sql.functions import dayofweek, when, avg

sessions_daytype_df = df.withColumn(
    "day_type",
    when(
        dayofweek("session_start_ts").isin(1, 7),
        "Weekend"
    ).otherwise("Weekday")
)

sessions_daytype_df.groupBy(
    "day_type"
).agg(
    avg("watch_minutes").alias("avg_watch_minutes")
).show()


+--------+------------------+
|day_type| avg_watch_minutes|
+--------+------------------+
| Weekday| 45.26460823373174|
| Weekend|45.411601967328295|
+--------+------------------+



In [22]:
from pyspark.sql.functions import sum, dense_rank
from pyspark.sql.window import Window

genre_content_df = sessions_df.join(
    content_df,
    "content_id",
    "inner"
).groupBy(
    "genre",
    "content_title"
).agg(
    sum("watch_minutes").alias("total_watch_minutes")
)

window_spec = Window.partitionBy(
    "genre"
).orderBy(
    col("total_watch_minutes").desc()
)

top_titles_df = genre_content_df.withColumn(
    "genre_rank",
    dense_rank().over(window_spec)
).filter(
    col("genre_rank") <= 3
)

top_titles_df.orderBy(
    "genre",
    "genre_rank"
).show(truncate=False)

+-----------+----------------+-------------------+----------+
|genre      |content_title   |total_watch_minutes|genre_rank|
+-----------+----------------+-------------------+----------+
|Adventure  |Infinite Horizon|66395              |1         |
|Adventure  |Lost Journey    |61823              |2         |
|Adventure  |Solar Memory    |60540              |3         |
|Animation  |Last River      |51398              |1         |
|Animation  |Emerald Circuit |51310              |2         |
|Animation  |Golden Puzzle   |50515              |3         |
|Comedy     |Blue Horizon    |61428              |1         |
|Comedy     |Bright Valley   |60921              |2         |
|Comedy     |Silver Valley   |60422              |3         |
|Documentary|Fading Signal   |47085              |1         |
|Documentary|Emerald Valley  |40819              |2         |
|Documentary|Solar Horizon   |36152              |3         |
|Drama      |Crimson Summit  |65724              |1         |
|Drama  

In [23]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

recent_session_window = Window.partitionBy(
    "subscriber_id"
).orderBy(
    col("session_start_ts").desc(),
    col("session_id").desc()
)

latest_sessions = df.filter(
    col("subscriber_id").isNotNull()
).withColumn(
    "rn",
    row_number().over(recent_session_window)
).filter(
    col("rn") == 1
).drop("rn")

latest_sessions.show(truncate=False)

+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+-------------------+
|session_id|subscriber_id|content_id|device_id|session_start      |watch_minutes|completion_pct|buffering_seconds|stream_quality|session_status|data_used_mb|user_rating|session_start_ts   |
+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+-------------------+
|WS00020822|SUB00001     |CT0073    |DV03     |2026-09-11 19:19:00|34           |89.5          |145.0            |HD            |Interrupted   |418.3       |NULL       |2026-09-11 19:19:00|
|WS00032744|SUB00002     |CT0076    |DV05     |2026-09-02 05:41:00|45           |73.8          |219.0            |Full HD       |Interrupted   |1184.1      |NULL       |2026-09-02 05:41:00|
|WS00006110|SUB00003     |CT0004    |DV03     |202

In [24]:
from pyspark.sql.functions import lag
from pyspark.sql.window import Window

subscriber_window = Window.partitionBy(
    "subscriber_id"
).orderBy(
    "session_start_ts",
    "session_id"
)

watch_comparison_df = df.filter(
    col("subscriber_id").isNotNull()
).withColumn(
    "previous_watch_minutes",
    lag("watch_minutes").over(subscriber_window)
).withColumn(
    "watch_minutes_difference",
    col("watch_minutes") - col("previous_watch_minutes")
)

watch_comparison_df.select(
    "subscriber_id",
    "session_id",
    "session_start_ts",
    "watch_minutes",
    "previous_watch_minutes",
    "watch_minutes_difference"
).show(truncate=False)

+-------------+----------+-------------------+-------------+----------------------+------------------------+
|subscriber_id|session_id|session_start_ts   |watch_minutes|previous_watch_minutes|watch_minutes_difference|
+-------------+----------+-------------------+-------------+----------------------+------------------------+
|SUB00001     |WS00064390|2026-01-02 17:32:00|58           |NULL                  |NULL                    |
|SUB00001     |WS00045183|2026-01-05 04:51:00|78           |58                    |20                      |
|SUB00001     |WS00028484|2026-01-09 08:46:00|4            |78                    |-74                     |
|SUB00001     |WS00043555|2026-01-24 19:41:00|108          |4                     |104                     |
|SUB00001     |WS00041862|2026-02-18 07:43:00|79           |108                   |-29                     |
|SUB00001     |WS00059616|2026-02-25 07:38:00|18           |79                    |-61                     |
|SUB00001     |WS00

In [25]:
from pyspark.sql.functions import dayofweek, when, avg

sessions_daytype_df = df.withColumn(
    "day_type",
    when(
        dayofweek("session_start_ts").isin(1, 7),
        "Weekend"
    ).otherwise("Weekday")
)

sessions_daytype_df.groupBy(
    "day_type"
).agg(
    avg("watch_minutes").alias("avg_watch_minutes")
).show()

+--------+------------------+
|day_type| avg_watch_minutes|
+--------+------------------+
| Weekday| 45.26460823373174|
| Weekend|45.411601967328295|
+--------+------------------+



In [27]:
combined_sessions_df = sessions_df.unionByName(
    extra_df
)

total_rows_before = combined_sessions_df.count()

deduplicated_sessions_df = combined_sessions_df.dropDuplicates(
    ["session_id"]
)

total_rows_after = deduplicated_sessions_df.count()

print("Rows Before Deduplication:", total_rows_before)
print("Rows After Deduplication:", total_rows_after)
print("Duplicate Rows Removed:",
      total_rows_before - total_rows_after)

Rows Before Deduplication: 90000
Rows After Deduplication: 89400
Duplicate Rows Removed: 600


In [28]:
from pyspark.sql.functions import col, sum

pipeline_df = (
    sessions_df
    .filter(col("session_status") == "Completed")
    .join(content_df, "content_id", "inner")
    .withColumn(
        "watch_hours",
        col("watch_minutes") / 60
    )
    .groupBy("genre")
    .agg(
        sum("watch_minutes").alias("total_watch_minutes")
    )
    .orderBy(col("total_watch_minutes").desc())
)

pipeline_df.explain(True)

pipeline_df.show()

== Parsed Logical Plan ==
'Sort ['total_watch_minutes DESC NULLS LAST], true
+- Aggregate [genre#100], [genre#100, sum(watch_minutes#22) AS total_watch_minutes#1142L]
   +- Project [content_id#19, session_id#17, subscriber_id#18, device_id#20, session_start#21, watch_minutes#22, completion_pct#23, buffering_seconds#24, stream_quality#25, session_status#26, data_used_mb#27, user_rating#28, content_title#99, genre#100, content_type#101, duration_minutes#102, release_year#103, content_language#104, (cast(watch_minutes#22 as double) / cast(60 as double)) AS watch_hours#1141]
      +- Project [content_id#19, session_id#17, subscriber_id#18, device_id#20, session_start#21, watch_minutes#22, completion_pct#23, buffering_seconds#24, stream_quality#25, session_status#26, data_used_mb#27, user_rating#28, content_title#99, genre#100, content_type#101, duration_minutes#102, release_year#103, content_language#104]
         +- Join Inner, (content_id#19 = content_id#98)
            :- Filter (sessio

In [29]:
import time
from pyspark.sql.functions import avg

completed_sessions_df = sessions_df.filter(
    col("session_status") == "Completed"
).cache()

start = time.time()
completed_sessions_df.count()
print("First Count:", time.time() - start)

start = time.time()
completed_sessions_df.count()
print("Second Count:", time.time() - start)

start = time.time()
completed_sessions_df.agg(
    avg("watch_minutes")
).show()
print("First Aggregation:", time.time() - start)

start = time.time()
completed_sessions_df.agg(
    avg("watch_minutes")
).show()
print("Second Aggregation:", time.time() - start)

First Count: 1.3503992557525635
Second Count: 0.18854522705078125
+------------------+
|avg(watch_minutes)|
+------------------+
| 76.90847934918648|
+------------------+

First Aggregation: 0.8409438133239746
+------------------+
|avg(watch_minutes)|
+------------------+
| 76.90847934918648|
+------------------+

Second Aggregation: 0.4245145320892334


In [30]:
from pyspark.storagelevel import StorageLevel

sessions_4k_df = sessions_df.filter(
    col("stream_quality") == "4K"
).persist(StorageLevel.MEMORY_AND_DISK)

sessions_4k_df.count()

sessions_4k_df.groupBy(
    "session_status"
).count().show()

sessions_4k_df.unpersist()

+--------------+-----+
|session_status|count|
+--------------+-----+
|     Completed|  599|
|     Abandoned| 1256|
|   Interrupted| 5457|
+--------------+-----+



DataFrame[session_id: string, subscriber_id: string, content_id: string, device_id: string, session_start: timestamp, watch_minutes: int, completion_pct: double, buffering_seconds: double, stream_quality: string, session_status: string, data_used_mb: double, user_rating: double]

In [31]:
from pyspark.sql.functions import spark_partition_id

sessions_8part_df = sessions_df.repartition(8)

print("Number of Partitions:",
      sessions_8part_df.rdd.getNumPartitions())

sessions_8part_df.groupBy(
    spark_partition_id().alias("partition_id")
).count().orderBy(
    "partition_id"
).show(truncate=False)

Number of Partitions: 8
+------------+-----+
|partition_id|count|
+------------+-----+
|0           |10000|
|1           |9999 |
|2           |10001|
|3           |10000|
|4           |10000|
|5           |10000|
|6           |10000|
|7           |10000|
+------------+-----+



In [32]:
sessions_8part_df = sessions_df.repartition(8)

coalesce_df = sessions_8part_df.coalesce(2)

print("Partitions:",
      coalesce_df.rdd.getNumPartitions())

coalesce_df.explain(True)

repartition_df = sessions_8part_df.repartition(2)

print("Partitions:",
      repartition_df.rdd.getNumPartitions())

repartition_df.explain(True)


Partitions: 2
== Parsed Logical Plan ==
Repartition 2, false
+- Repartition 8, true
   +- Relation [session_id#17,subscriber_id#18,content_id#19,device_id#20,session_start#21,watch_minutes#22,completion_pct#23,buffering_seconds#24,stream_quality#25,session_status#26,data_used_mb#27,user_rating#28] csv

== Analyzed Logical Plan ==
session_id: string, subscriber_id: string, content_id: string, device_id: string, session_start: timestamp, watch_minutes: int, completion_pct: double, buffering_seconds: double, stream_quality: string, session_status: string, data_used_mb: double, user_rating: double
Repartition 2, false
+- Repartition 8, true
   +- Relation [session_id#17,subscriber_id#18,content_id#19,device_id#20,session_start#21,watch_minutes#22,completion_pct#23,buffering_seconds#24,stream_quality#25,session_status#26,data_used_mb#27,user_rating#28] csv

== Optimized Logical Plan ==
Repartition 2, false
+- Repartition 8, true
   +- Relation [session_id#17,subscriber_id#18,content_id#19,d

In [34]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.storagelevel import StorageLevel

#1
sessions_df = (
    sessions_df
    .unionByName(extra_df)
    .dropDuplicates(["session_id"])
)

# 2
clean_sessions_df = (
    sessions_df
    .na.fill({
        "buffering_seconds": 0,
        "data_used_mb": 0,
        "watch_minutes": 0
    })
)

# 3
enriched_df = (
    clean_sessions_df

    .join(
        subscribers_df,
        "subscriber_id",
        "left"
    )

    .join(
        plans_df,
        "plan_id",
        "left"
    )

    .join(
        content_df,
        "content_id",
        "left"
    )

    .join(
        devices_df,
        "device_id",
        "left"
    )

    .withColumn(
        "session_ts",
        to_timestamp("session_start")
    )

    .withColumn(
        "month",
        date_format("session_ts", "yyyy-MM")
    )

    .withColumn(
        "watch_hours",
        col("watch_minutes") / 60
    )
)

# 4
monthly_report_df = (
    enriched_df
    .groupBy(
        "month",
        "region",
        "genre"
    )
    .agg(
        count_distinct("session_id")
            .alias("session_count"),

        round(
            sum("watch_hours"), 2
        ).alias("total_watch_hours"),

        round(
            (
                sum(
                    when(
                        col("session_status") == "Completed",
                        1
                    ).otherwise(0)
                )
                * 100.0
                / count("session_id")
            ),
            2
        ).alias("completion_rate_pct"),

        round(
            avg("buffering_seconds"),
            2
        ).alias("avg_buffering_seconds"),

        round(
            sum("data_used_mb"),
            2
        ).alias("total_data_usage_mb")
    )
    .orderBy(
        "month",
        "region",
        "genre"
    )
)

# 5
title_watch_df = (
    enriched_df
    .groupBy(
        "genre",
        "content_title"
    )
    .agg(
        round(
            sum("watch_hours"),
            2
        ).alias("watch_hours")
    )
)

genre_window = Window.partitionBy(
    "genre"
).orderBy(
    col("watch_hours").desc()
)

top3_titles_df = (
    title_watch_df
    .withColumn(
        "genre_rank",
        dense_rank().over(genre_window)
    )
    .filter(
        col("genre_rank") <= 3
    )
    .orderBy(
        "genre",
        "genre_rank"
    )
)

# 6
final_report_df = monthly_report_df.cache()

print("Report Record Count:")
print(final_report_df.count())

final_report_df.show(20, truncate=False)

final_report_df.unpersist()

print("Top 3 Titles Within Each Genre")
top3_titles_df.show(truncate=False)

# 7
(
    final_report_df
    .repartition(3)
    .write
    .mode("overwrite")
    .parquet("streaming_performance_report.csv")
)

Report Record Count:
432
+-------+-------+-----------+-------------+-----------------+-------------------+---------------------+-------------------+
|month  |region |genre      |session_count|total_watch_hours|completion_rate_pct|avg_buffering_seconds|total_data_usage_mb|
+-------+-------+-----------+-------------+-----------------+-------------------+---------------------+-------------------+
|2026-01|NULL   |Adventure  |37           |38.35            |8.11               |33.38                |44063.6            |
|2026-01|NULL   |Animation  |21           |13.02            |4.76               |26.86                |13333.2            |
|2026-01|NULL   |Comedy     |9            |4.42             |0.0                |30.78                |5805.3             |
|2026-01|NULL   |Documentary|22           |11.67            |9.09               |40.86                |13426.2            |
|2026-01|NULL   |Drama      |23           |17.22            |8.7                |32.61                |1518